In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

ONLINE_STORE_KEY = 0
REPORTING_CURRENCY = "EUR"

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


def us_date(column_name):
    return F.to_date(F.try_to_timestamp(F.trim(F.col(column_name)), F.lit("M/d/yyyy")))


def write_silver(df, table_name):
    (
        df.withColumn("_silver_processed_at_utc", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

In [ ]:
#type
bronze_df = spark.table("bronze.orders_raw")

typed_df = bronze_df.select(
    F.trim("order_number").try_cast("int").alias("order_number"),
    F.trim("line_item").try_cast("int").alias("line_item"),
    us_date("order_date_raw").alias("order_date"),
    us_date("delivery_date_raw").alias("delivery_date"),
    F.trim("customer_key").try_cast("int").alias("customer_key"),
    F.trim("store_key").try_cast("int").alias("store_key"),
    F.trim("product_key").try_cast("int").alias("product_key"),
    F.trim("quantity_raw").try_cast("int").alias("quantity"),
    F.upper(F.trim("currency_code_raw")).alias("currency_code"),
    "order_date_raw",
    "delivery_date_raw",
    "quantity_raw",
    "_batch_id",
    "_source_file",
    "_ingested_at_utc",
)

#deduplicate deterministically: latest ingestion wins, business columns break ties
dedup_window = Window.partitionBy("order_number", "line_item").orderBy(
    F.col("_ingested_at_utc").desc(),
    F.col("_batch_id").desc(),
    "order_date_raw", "delivery_date_raw", "customer_key", "store_key",
    "product_key", "quantity_raw", "currency_code",
)
typed_df = typed_df.withColumn("_duplicate_rank", F.row_number().over(dedup_window))

In [ ]:
#enrich: reference keys, product price and order-date FX (ADR 003)
customers_df = spark.table("silver.customers").select("customer_key", F.lit(True).alias("_customer_found"))
stores_df = spark.table("silver.stores").select("store_key", F.lit(True).alias("_store_found"))
products_df = spark.table("silver.products").select("product_key", "unit_price_usd")

daily_fx_df = spark.table("silver.exchange_rates_daily")
order_fx_df = daily_fx_df.select(
    F.col("rate_date").alias("order_date"),
    "currency_code",
    F.col("rate_per_usd").alias("order_rate_per_usd"),
    F.col("source_rate_date").alias("order_fx_rate_date"),
)
eur_fx_df = daily_fx_df.filter(F.col("currency_code") == REPORTING_CURRENCY).select(
    F.col("rate_date").alias("order_date"),
    F.col("rate_per_usd").alias("eur_rate_per_usd"),
    F.col("source_rate_date").alias("eur_fx_rate_date"),
)

enriched_df = (
    typed_df
    .join(customers_df, "customer_key", "left")
    .join(stores_df, "store_key", "left")
    .join(products_df, "product_key", "left")
    .join(order_fx_df, ["order_date", "currency_code"], "left")
    .join(eur_fx_df, "order_date", "left")
    .withColumn("is_online_order", F.col("store_key") == ONLINE_STORE_KEY)
    .withColumn(
        "unit_price_local",
        F.round(F.col("unit_price_usd") * F.col("order_rate_per_usd"), 2).cast("decimal(18,2)"),
    )
    .withColumn(
        "sales_amount_local",
        (F.col("quantity") * F.col("unit_price_local")).cast("decimal(18,2)"),
    )
    .withColumn(
        "sales_amount_eur",
        F.round(
            F.col("sales_amount_local") / F.col("order_rate_per_usd") * F.col("eur_rate_per_usd"), 2
        ).cast("decimal(18,2)"),
    )
)

In [ ]:
#data-quality rules: a row failing any rule is quarantined with every reason it failed
CHECKS = {
    "duplicate_order_line": F.col("_duplicate_rank") > 1,
    "invalid_order_key": F.col("order_number").isNull() | F.col("line_item").isNull(),
    "invalid_order_date": F.col("order_date").isNull(),
    "invalid_delivery_date": F.col("delivery_date_raw").isNotNull() & F.col("delivery_date").isNull(),
    "missing_online_delivery_date": (F.col("store_key") == ONLINE_STORE_KEY) & F.col("delivery_date_raw").isNull(),
    "unexpected_store_delivery_date": (F.col("store_key") != ONLINE_STORE_KEY) & F.col("delivery_date_raw").isNotNull(),
    "delivery_before_order": F.col("delivery_date") < F.col("order_date"),
    "invalid_quantity": F.col("quantity").isNull() | (F.col("quantity") <= 0),
    "unknown_customer": F.col("_customer_found").isNull(),
    "unknown_store": F.col("_store_found").isNull(),
    "unknown_product": F.col("unit_price_usd").isNull(),
    "missing_fx_rate": F.col("order_rate_per_usd").isNull() | F.col("eur_rate_per_usd").isNull(),
}

checked_df = enriched_df.withColumn(
    "failure_reasons",
    F.array_compact(F.array(*[
        F.when(F.coalesce(failed, F.lit(False)), F.lit(name)) for name, failed in CHECKS.items()
    ])),
)

SALES_COLUMNS = [
    "order_number", "line_item", "order_date", "delivery_date",
    "customer_key", "store_key", "is_online_order", "product_key", "quantity",
    "currency_code", "unit_price_usd",
    "order_rate_per_usd", "order_fx_rate_date", "eur_rate_per_usd", "eur_fx_rate_date",
    "unit_price_local", "sales_amount_local", "sales_amount_eur",
    "_batch_id", "_source_file", "_ingested_at_utc",
]

write_silver(
    checked_df.filter(F.size("failure_reasons") == 0).select(*SALES_COLUMNS),
    "silver.sales_lines",
)
write_silver(
    checked_df.filter(F.size("failure_reasons") > 0).drop("_customer_found", "_store_found"),
    "silver.sales_lines_quarantine",
)

In [ ]:
#validate and reconcile with Bronze
bronze_rows = bronze_df.count()
sales_df = spark.table("silver.sales_lines")
quarantine_df = spark.table("silver.sales_lines_quarantine")
silver_rows = sales_df.count()
quarantine_rows = quarantine_df.count()

bronze_quantity = bronze_df.select(F.sum(F.trim("quantity_raw").try_cast("int"))).first()[0]
silver_quantity = sales_df.select(F.sum("quantity")).first()[0] or 0
quarantine_quantity = quarantine_df.select(F.sum("quantity")).first()[0] or 0

duplicate_keys = sales_df.groupBy("order_number", "line_item").count().filter("count > 1").count()

print(f"Bronze rows: {bronze_rows:,}")
print(f"Silver rows: {silver_rows:,}")
print(f"Quarantined rows: {quarantine_rows:,}")
print(f"Bronze quantity: {bronze_quantity:,} = Silver {silver_quantity:,} + quarantine {quarantine_quantity:,}")
print(f"Duplicate keys in Silver: {duplicate_keys:,}")

assert silver_rows + quarantine_rows == bronze_rows
assert silver_quantity + quarantine_quantity == bronze_quantity
assert duplicate_keys == 0

display(
    quarantine_df.select(F.explode("failure_reasons").alias("failure_reason"))
    .groupBy("failure_reason").count()
)
display(
    sales_df.groupBy("currency_code").agg(
        F.count("*").alias("lines"),
        F.sum("sales_amount_local").alias("sales_amount_local"),
        F.sum("sales_amount_eur").alias("sales_amount_eur"),
    ).orderBy("currency_code")
)